In [33]:
"""
RAG Ingestion Pipeline — Local SLM Variant
===========================================

Identical to the Groq/cloud variant, with one change: metadata extraction
runs against a local Small Language Model served by Ollama instead of the
Groq API.  No API key is required.  No rate limiting.  No network dependency
beyond the initial model pull.

The pipeline performs the following steps:

1. PDF parsing with Docling (layout-aware document conversion).
2. Semantic chunking with HybridChunker backed by a HuggingFace tokenizer.
3. Metadata extraction with a local SLM via Ollama (OpenAI-compatible API).
4. Embedding generation with a HuggingFace sentence-transformers model.
5. Qdrant vector storage and querying.

Setup — run once in your terminal:
    # Install Ollama: https://ollama.com/download
    ollama pull qwen2.5:0.5b    # ~397 MB, fast on CPU
    ollama serve                # starts the local API on http://localhost:11434

Environment variables (.env):
    OLLAMA_BASE_URL   Optional. Defaults to "http://localhost:11434/v1".
    OLLAMA_MODEL      Optional. Defaults to "phi3.5".
    EMBEDDING_MODEL   Optional. Defaults to "sentence-transformers/all-MiniLM-L6-v2".
    QDRANT_URL        Qdrant Cloud cluster URL.
    QDRANT_API_KEY    Qdrant Cloud API key.
    COLLECTION_NAME   Optional. Defaults to "rag_collection".

Requirements (install via pip):
    docling
    qdrant-client
    sentence-transformers
    transformers
    python-dotenv
    openai            # used as the Ollama client (OpenAI-compatible)
"""

'\nRAG Ingestion Pipeline — Local SLM Variant\n===========================================\n\nIdentical to the Groq/cloud variant, with one change: metadata extraction\nruns against a local Small Language Model served by Ollama instead of the\nGroq API.  No API key is required.  No rate limiting.  No network dependency\nbeyond the initial model pull.\n\nThe pipeline performs the following steps:\n\n1. PDF parsing with Docling (layout-aware document conversion).\n2. Semantic chunking with HybridChunker backed by a HuggingFace tokenizer.\n3. Metadata extraction with a local SLM via Ollama (OpenAI-compatible API).\n4. Embedding generation with a HuggingFace sentence-transformers model.\n5. Qdrant vector storage and querying.\n\nSetup — run once in your terminal:\n    # Install Ollama: https://ollama.com/download\n    ollama pull qwen2.5:0.5b    # ~397 MB, fast on CPU\n    ollama serve                # starts the local API on http://localhost:11434\n\nEnvironment variables (.env):\n    OLL

In [34]:
from __future__ import annotations

import logging
import os
from typing import Optional

from dotenv import load_dotenv

# ---------------------------------------------------------------------------
# Logging configuration
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("RAG Pipeline — Local SLM")

# ---------------------------------------------------------------------------
# Environment loading
# ---------------------------------------------------------------------------
load_dotenv()

# Local SLM via Ollama — no API key required
OLLAMA_BASE_URL: str = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
OLLAMA_MODEL: str = os.getenv("OLLAMA_MODEL", "qwen2.5:0.5b")

EMBEDDING_MODEL_NAME: str = os.getenv(
    "EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
)
QDRANT_URL: str = os.getenv("QDRANT_URL")
QDRANT_API_KEY: str = os.getenv("QDRANT_API_KEY")
COLLECTION_NAME: str = os.getenv("COLLECTION_NAME", "rag_collection")
CHUNKER_TOKENIZER_MODEL: str = "sentence-transformers/all-MiniLM-L6-v2"


In [35]:
# ---------------------------------------------------------------------------
# Data containers
# ---------------------------------------------------------------------------
import hashlib
import uuid
from dataclasses import dataclass, field
from typing import Any, Dict, List

@dataclass
class Chunk:
    """Represents a semantically meaningful chunk of a document."""

    text: str
    index: int
    source_file: str
    page_numbers: List[int] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)
    embedding: Optional[List[float]] = None
    chunk_id: str = field(default_factory=lambda: str(uuid.uuid4()))

    def to_point(self) -> Dict[str, Any]:
        """Convert the chunk into a Qdrant point payload structure."""
        return {
            "id": self.chunk_id,
            "vector": self.embedding,
            "payload": {
                "text": self.text,
                "index": self.index,
                "source_file": self.source_file,
                "page_numbers": self.page_numbers,
                "metadata": self.metadata,
            },
        }


In [36]:
# ---------------------------------------------------------------------------
# PDF parsing with Docling
# ---------------------------------------------------------------------------
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from pathlib import Path


class PDFParser:
    """Wraps Docling to convert PDF files into structured document objects."""

    def __init__(self) -> None:
        pipeline_options = PdfPipelineOptions()
        pipeline_options.do_ocr = False
        pipeline_options.do_table_structure = False
        pipeline_options.do_code_enrichment = False
        pipeline_options.do_formula_enrichment = False

        self._converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=pipeline_options,
                    backend=PyPdfiumDocumentBackend,
                )
            }
        )
        logging.info("PDFParser initialized (fast text-only mode).")

    def parse(self, pdf_path: Path):
        if not pdf_path.exists():
            raise FileNotFoundError(f"PDF file not found: {pdf_path}")
        logging.info("Parsing PDF: %s", pdf_path)
        return self._converter.convert(str(pdf_path)).document


In [37]:
# ---------------------------------------------------------------------------
# Semantic chunking with HybridChunker
# ---------------------------------------------------------------------------
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer
from tqdm import tqdm

MAX_TOKENS = 512


class SemanticChunker:
    """Chunks text using HybridChunker with a HuggingFace tokenizer."""

    def __init__(
        self,
        tokenizer_model: str = EMBEDDING_MODEL_NAME,
        chunk_size: int = 512,
    ) -> None:
        self._hf_tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
        hf_tokenizer = HuggingFaceTokenizer(
            tokenizer=self._hf_tokenizer,
            max_tokens=chunk_size,
        )
        self._chunker = HybridChunker(
            tokenizer=hf_tokenizer, max_tokens=chunk_size, merge_peers=True
        )
        logging.info(
            f"SemanticChunker initialized (model={tokenizer_model}, chunk_size={chunk_size})."
        )

    def _truncate(self, text: str) -> str:
        tokens = self._hf_tokenizer.encode(text, add_special_tokens=True)
        if len(tokens) <= MAX_TOKENS:
            return text
        return self._hf_tokenizer.decode(tokens[:MAX_TOKENS], skip_special_tokens=True)

    def chunk(self, document, source_file: str) -> list[Chunk]:
        logging.info("Chunking document from %s.", source_file)
        chunks: list[Chunk] = []
        for idx, dl_chunk in tqdm(enumerate(self._chunker.chunk(document)), desc="Chunking", unit="chunk"):
            text = dl_chunk.text if hasattr(dl_chunk, "text") else str(dl_chunk)
            chunks.append(
                Chunk(text=self._truncate(text.strip()), index=idx, source_file=source_file)
            )
        logging.info("Produced %d chunks.", len(chunks))
        return chunks


In [38]:
# ---------------------------------------------------------------------------
# Metadata extraction with a local SLM via Ollama
# ---------------------------------------------------------------------------
# Ollama exposes an OpenAI-compatible REST API, so the openai package is the
# client.  No API key is needed; the placeholder string satisfies the client's
# non-empty key validation without being sent anywhere.
#
# Prerequisites (run once in a terminal before executing this notebook):
#   ollama pull qwen2.5:0.5b  # downloads the model (~397 MB)
#   ollama serve            # starts the API on http://localhost:11434
# ---------------------------------------------------------------------------
import json
from openai import OpenAI
from tqdm import tqdm


class MetadataExtractor:
    """Extracts structured metadata from text chunks using a local SLM via Ollama.

    Ollama's OpenAI-compatible endpoint accepts the same chat/completions
    request format as any cloud provider.  Because inference is local there is
    no rate limiting, so requests run sequentially to avoid overwhelming the
    single-process Ollama server on CPU hardware.
    """

    _SYSTEM_PROMPT = (
        "You are a metadata extraction assistant. "
        "Return ONLY a JSON object with keys: "
        "summary (string), topics (list), entities (list), keywords (list)."
    )

    def __init__(
        self,
        base_url: str = OLLAMA_BASE_URL,
        model: str = OLLAMA_MODEL,
        max_tokens: int = 512,
        temperature: float = 0.0,
    ) -> None:
        # api_key is required by the openai client but ignored by Ollama
        self._client = OpenAI(base_url=base_url, api_key="ollama")
        self._model = model
        self._max_tokens = max_tokens
        self._temperature = temperature
        logger.info("MetadataExtractor initialized (model=%s, backend=ollama).", model)

    @staticmethod
    def check_ollama() -> None:
        """Verify Ollama is reachable before starting a batch."""
        import urllib.request
        try:
            urllib.request.urlopen("http://localhost:11434", timeout=3)
        except Exception as exc:
            raise RuntimeError(
                "Ollama is not running. Start it with: ollama serve"
            ) from exc

    def extract(self, chunk: "Chunk") -> Dict[str, Any]:
        """Extract metadata for a single chunk."""
        import re
        try:
            response = self._client.chat.completions.create(
                model=self._model,
                messages=[
                    {"role": "system", "content": self._SYSTEM_PROMPT},
                    {"role": "user", "content": chunk.text},
                ],
                max_tokens=self._max_tokens,
                temperature=self._temperature,
            )
            raw = response.choices[0].message.content or ""
            # Try direct parse first, then extract first JSON object from prose
            try:
                metadata = json.loads(raw)
            except json.JSONDecodeError:
                match = re.search(r"\{.*\}", raw, re.DOTALL)
                metadata = json.loads(match.group()) if match else {}
        except Exception as exc:
            logger.warning("Metadata extraction failed for chunk %d: %s.", chunk.index, exc)
            metadata = {}

        metadata.setdefault("summary", "")
        metadata.setdefault("topics", [])
        metadata.setdefault("entities", [])
        metadata.setdefault("keywords", [])
        return metadata

    def extract_batch(self, chunks: list["Chunk"]) -> None:
        """Extract metadata for all chunks sequentially, mutating chunk.metadata in place.

        Sequential (not parallel) because Ollama runs a single model process
        locally — concurrent requests queue inside Ollama anyway and add thread
        overhead without improving throughput on CPU.
        """
        for chunk in tqdm(chunks, desc="Metadata", unit="chunk"):
            chunk.metadata = self.extract(chunk)


In [39]:
# ---------------------------------------------------------------------------
# Embedding generation
# ---------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer


class Embedder:
    """Generates dense vector embeddings for text chunks."""

    def __init__(self, model_name: str = EMBEDDING_MODEL_NAME) -> None:
        self._model = SentenceTransformer(model_name)
        self._dimension = self._model.get_embedding_dimension()
        logger.info(
            "Embedder initialized (model=%s, dimension=%d).",
            model_name,
            self._dimension,
        )

    @property
    def dimension(self) -> int:
        return self._dimension

    def embed(self, texts: List[str]) -> List[List[float]]:
        if not texts:
            return []
        vectors = self._model.encode(
            texts,
            convert_to_numpy=True,
            show_progress_bar=True,
            batch_size=32,
        )
        return [vec.tolist() for vec in vectors]


In [40]:
# ---------------------------------------------------------------------------
# Qdrant vector store
# ---------------------------------------------------------------------------
from qdrant_client import QdrantClient
from qdrant_client.http import models as qdrant_models


class VectorStore:
    """Manages a Qdrant Cloud collection for storage and retrieval."""

    def __init__(
        self,
        url: str = QDRANT_URL,
        api_key: str = QDRANT_API_KEY,
        collection_name: str = COLLECTION_NAME,
        vector_size: int = 384,
        distance: str = "Cosine",
    ) -> None:
        self._qdrant_models = qdrant_models
        self._collection_name = collection_name
        self._client = QdrantClient(url=url, api_key=api_key)
        self._ensure_collection(vector_size=vector_size, distance=distance)
        logger.info(
            "VectorStore initialized (url=%s, collection=%s, dim=%d).",
            url, collection_name, vector_size,
        )

    def _ensure_collection(self, vector_size: int, distance: str) -> None:
        existing_names = {c.name for c in self._client.get_collections().collections}
        if self._collection_name not in existing_names:
            self._client.create_collection(
                collection_name=self._collection_name,
                vectors_config=self._qdrant_models.VectorParams(
                    size=vector_size, distance=distance,
                ),
            )
            logger.info("Created Qdrant collection: %s", self._collection_name)

    def upsert_chunks(self, chunks: List[Chunk]) -> None:
        if not chunks:
            return
        points = [c.to_point() for c in chunks if c.embedding is not None]
        if not points:
            return
        self._client.upsert(
            collection_name=self._collection_name, points=points, wait=True,
        )
        logger.info("Upserted %d points into %s.", len(points), self._collection_name)

    def query(
        self,
        query_vector: List[float],
        top_k: int = 5,
        filters: Optional[Dict[str, Any]] = None,
    ) -> List[Dict[str, Any]]:
        must_filters = []
        if filters:
            for key, value in filters.items():
                must_filters.append(
                    self._qdrant_models.FieldCondition(
                        key=key,
                        match=self._qdrant_models.MatchValue(value=value),
                    )
                )
        query_filter = (
            self._qdrant_models.Filter(must=must_filters) if must_filters else None
        )
        results = self._client.query_points(
            collection_name=self._collection_name,
            query=query_vector,
            limit=top_k,
            query_filter=query_filter,
        )
        return [
            {"id": str(p.id), "score": float(p.score), "payload": p.payload or {}}
            for p in results.points
        ]


In [41]:
# ---------------------------------------------------------------------------
# Pipeline orchestration
# ---------------------------------------------------------------------------
import logging
import hashlib
import uuid


class RAGPipeline:
    """Orchestrates the full RAG ingestion and querying pipeline."""

    def __init__(
        self,
        chunk_size: int = 512,
        enable_metadata_extraction: bool = True,
    ) -> None:
        self.parser = PDFParser()
        self.chunker = SemanticChunker(chunk_size=chunk_size)
        self.embedder = Embedder()
        self.store = VectorStore(vector_size=self.embedder.dimension)
        self.enable_metadata_extraction = enable_metadata_extraction
        if enable_metadata_extraction:
            MetadataExtractor.check_ollama()
        self.extractor = MetadataExtractor() if enable_metadata_extraction else None

    def _stable_chunk_id(self, chunk: Chunk) -> str:
        digest = hashlib.sha256(
            f"{chunk.source_file}:{chunk.index}:{chunk.text}".encode("utf-8")
        ).hexdigest()
        return str(uuid.UUID(digest[:32].ljust(32, "0")))

    def ingest(self, pdf_path: Path) -> list[Chunk]:
        logging.info("=== Starting ingestion for: %s ===", pdf_path.name)
        document = self.parser.parse(pdf_path)
        chunks = self.chunker.chunk(document, source_file=pdf_path.name)

        if not chunks:
            logging.warning("No chunks produced from %s. Skipping.", pdf_path.name)
            return []

        for chunk in chunks:
            chunk.chunk_id = self._stable_chunk_id(chunk)

        if self.extractor is not None:
            logging.info("Extracting metadata for %d chunks...", len(chunks))
            self.extractor.extract_batch(chunks)

        logging.info("Generating embeddings for %d chunks...", len(chunks))
        embeddings = self.embedder.embed([c.text for c in chunks])
        for chunk, emb in zip(chunks, embeddings):
            chunk.embedding = emb

        self.store.upsert_chunks(chunks)
        logging.info("=== Ingestion complete for: %s ===", pdf_path.name)
        return chunks

    def query(
        self, query_text: str, top_k: int = 5, filters: dict | None = None
    ) -> list[dict]:
        logging.info("Querying: %r (top_k=%d)", query_text, top_k)
        query_vector = self.embedder.embed([query_text])[0]
        results = self.store.query(query_vector=query_vector, top_k=top_k, filters=filters)
        logging.info("Retrieved %d results.", len(results))
        return results


In [42]:
# Purpose: Initialize the RAGPipeline with the local Ollama SLM for metadata extraction.
# Prerequisites: Ollama must be running with phi3.5 pulled.
#   ollama pull phi3.5 && ollama serve

import logging
logging.getLogger().setLevel(logging.INFO)

pipeline = RAGPipeline(
    chunk_size=512,
    enable_metadata_extraction=True,
)

print("RAG Pipeline successfully initialized.")


2026-07-06 22:03:49 | INFO     | root | PDFParser initialized (fast text-only mode).
2026-07-06 22:03:49 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-06 22:03:49 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config.json "HTTP/1.1 200 OK"
2026-07-06 22:03:49 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-06 22:03:49 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-06 22:03:49 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/sent

RAG Pipeline successfully initialized.


In [46]:
# Purpose: Execute the end-to-end ingestion process for a specific PDF document.

target_pdf = Path("../ingestion-docs/kinesis-dg.pdf")

if not target_pdf.exists():
    print(f"Error: Could not find {target_pdf}. Please ensure the file is in the correct directory.")
else:
    print(f"Starting ingestion for {target_pdf.name}...")
    ingested_chunks = pipeline.ingest(pdf_path=target_pdf)
    print(f"Ingestion complete. Successfully processed and stored {len(ingested_chunks)} chunks.")


2026-07-07 06:00:40 | INFO     | root | === Starting ingestion for: kinesis-dg.pdf ===
2026-07-07 06:00:40 | INFO     | root | Parsing PDF: ../ingestion-docs/kinesis-dg.pdf
2026-07-07 06:00:40 | INFO     | docling.datamodel.document | detected formats: [<InputFormat.PDF: 'pdf'>]
2026-07-07 06:00:40 | INFO     | docling.document_converter | Going to convert document batch...
2026-07-07 06:00:40 | INFO     | docling.pipeline.base_pipeline | Processing document kinesis-dg.pdf


Starting ingestion for kinesis-dg.pdf...


2026-07-07 06:37:30 | INFO     | docling.document_converter | Finished converting document kinesis-dg.pdf in 2210.17 sec.
2026-07-07 06:37:30 | INFO     | root | Chunking document from kinesis-dg.pdf.
/home/techmind-dev/engineering/ai-engineering/aws-ai-engineer-portfolio/.venv/lib/python3.12/site-packages/docling_core/types/doc/document.py:3021: UserWarning: Provenance bbox coordinate r on page 38 is outside page bounds: value=666.1900024414062 > hi=612.0; clamping to 612.0
  prov.bbox = cls._clamp_bbox_to_page(
/home/techmind-dev/engineering/ai-engineering/aws-ai-engineer-portfolio/.venv/lib/python3.12/site-packages/docling_core/types/doc/document.py:3021: UserWarning: Provenance bbox coordinate r on page 44 is outside page bounds: value=634.31201171875 > hi=612.0; clamping to 612.0
  prov.bbox = cls._clamp_bbox_to_page(
/home/techmind-dev/engineering/ai-engineering/aws-ai-engineer-portfolio/.venv/lib/python3.12/site-packages/docling_core/types/doc/document.py:3021: UserWarning: Prov

Ingestion complete. Successfully processed and stored 1177 chunks.


In [47]:
# Purpose: Verify the pipeline by querying the Qdrant vector store using natural language.

query_text = "What is kinesis?"
print(f"Executing Query: '{query_text}'\n")

search_results = pipeline.query(query_text=query_text, top_k=3)

for i, result in enumerate(search_results, start=1):
    score = result.get("score", 0.0)
    payload = result.get("payload", {})
    metadata = payload.get("metadata", {})
    text_preview = payload.get("text", "")

    print(f"--- Result {i} (Similarity Score: {score:.4f}) ---")
    print(f"Source File: {payload.get('source_file', 'Unknown')}")
    if metadata.get("summary"):
        print(f"Summary: {metadata['summary']}")
    print(f"Content Preview: {text_preview}\n")


2026-07-07 16:58:24 | INFO     | root | Querying: 'What is kinesis?' (top_k=3)


Executing Query: 'What is kinesis?'



Batches: 100%|██████████| 1/1 [00:00<00:00, 34.12it/s]
2026-07-07 16:58:24 | INFO     | httpx | HTTP Request: POST https://009fe11d-a542-4e58-9af7-73da25278046.sa-east-1-0.aws.cloud.qdrant.io:6333/collections/qdrant-rag-collection/points/query "HTTP/1.1 200 OK"
2026-07-07 16:58:24 | INFO     | root | Retrieved 3 results.


--- Result 1 (Similarity Score: 0.6766) ---
Source File: kinesis-dg.pdf
Summary: Archived topics for the Kinesis Client Library
Content Preview: The following topics have been archived . To see current Kinesis Client Library documentation , see Use Kinesis Client Library .

--- Result 2 (Similarity Score: 0.6609) ---
Source File: kinesis-dg.pdf
Summary: List of tags associated with the specified Kinesis resource.
Content Preview: Lists the tags for the specified Kinesis resource .

--- Result 3 (Similarity Score: 0.6494) ---
Source File: kinesis-dg.pdf
Summary: Configuration Properties for the Kinesis Client Library
Content Preview: You can set configuration properties to customize Kinesis Client Library ' s functionality to meet your specific requirements . The following table describes configuration properties and classes .

